In [13]:
# =====================================================================
# Dual-View Mammography Deep Learning Framework
# Task 1 — Characterization: Comprehensive Ablation Study Suite
# Core Backbone: EVA02-B (Unified Linear Evaluation Pipeline)
# Methodology: Strict Leakage-Free Nested Cross-Validation (Inner-Val)
# Performance Log: Explicit Inner-Val & Pristine Outer Test Streams
# =====================================================================

import os
import random
import warnings
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')

# ─── 1. GLOBAL CONFIGURATION & HYPERPARAMETERS ──────────────────────
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")

# Standardized publication-grade hyperparameters
IMG_SIZE = 448       # Restored to original EVA02 high-resolution target
BATCH_SIZE = 16      # Restored to original stable batch size
NUM_EPOCHS = 50      # Restored to original training horizon
WARMUP_EPOCHS = 3    # Restored backbone freezing warmup period
ES_START = 8         # Restored early stopping activation epoch
PATIENCE = 12        # Restored original convergence patience window
NUM_WORKERS = 4

# Pipeline data tracking paths
SPLIT_DIR = "./splits"
IMAGE_DIR = "./processed"  
CKPT_DIR  = "./checkpoints/ablation_suite"
STAT_DIR  = "./outputs/statistics"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(STAT_DIR, exist_ok=True)

# ─── 2. DATASET & TRANSFORMS ─────────────────────────────────────────
def get_transforms():
    train_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE), # Resolves 1024x1024 input shape mismatch
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=0, value=0),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    val_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE), 
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    return train_transform, val_transform

class DualViewDataset(Dataset):
    def __init__(self, df, transform=None, label_col='label_char'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.label_col = label_col
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        cc_img = Image.open(os.path.join(IMAGE_DIR, row['cc_path'])).convert('RGB')
        mlo_img = Image.open(os.path.join(IMAGE_DIR, row['mlo_path'])).convert('RGB')
        
        cc_np = np.array(cc_img)
        mlo_np = np.array(mlo_img)
        
        if self.transform:
            cc_np = self.transform(image=cc_np)['image']
            mlo_np = self.transform(image=mlo_np)['image']
            
        label = int(row[self.label_col])
        return {'cc': cc_np, 'mlo': mlo_np, 'label': label}

# ─── 3. MODEL ARCHITECTURES (STRUCTURAL ABLATION MATRIX) ──────────────
class CrossViewAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.scale = dim ** -0.5
        self.proj = nn.Linear(dim, dim)
        
    def forward(self, x1, x2):
        q = self.q(x1.unsqueeze(1))
        k = self.k(x2.unsqueeze(1))
        v = self.v(x2.unsqueeze(1))
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).squeeze(1)
        return x1 + self.proj(out)

class MammoXVAblationModel(nn.Module):
    def __init__(self, mode='Pyramid_CrossAttn_JointHead', num_classes=1):
        super().__init__()
        self.mode = mode
        
        # Core Foundation Encoder Base
        self.backbone = timm.create_model('eva02_base_patch14_448.mim_in22k_ft_in22k_in1k', pretrained=True, num_classes=0)
        dim = self.backbone.num_features
        
        # Pipeline Configuration Flags
        self.has_pyramid = 'No_Pyramid' not in mode and 'Single_Scale' not in mode
        self.has_attention = 'No_Attention' not in mode and 'Late_Fusion' not in mode
        self.is_hybrid_fusion = mode == 'Pyramid_CrossAttn_LateFusion'
        self.has_aux = 'No_Aux' not in mode or self.is_hybrid_fusion
        
        if self.has_attention or self.is_hybrid_fusion:
            self.attn_cc_to_mlo = CrossViewAttention(dim)
            self.attn_mlo_to_cc = CrossViewAttention(dim)
            
        if self.has_pyramid:
            self.pyramid_conv = nn.Linear(dim, dim)
            
        # Top-layer Classification Head Topologies
        if self.is_hybrid_fusion or mode == 'Late_Fusion':
            self.head_cc = nn.Linear(dim, num_classes)
            self.head_mlo = nn.Linear(dim, num_classes)
        else:
            self.classifier = nn.Linear(dim * 2, num_classes)
            if self.has_aux:
                self.aux_cc = nn.Linear(dim, num_classes)
                self.aux_mlo = nn.Linear(dim, num_classes)

    def forward(self, cc, mlo):
        f_cc = self.backbone(cc)
        f_mlo = self.backbone(mlo)
        
        if self.has_pyramid:
            f_cc = f_cc + torch.tanh(self.pyramid_conv(f_cc))
            f_mlo = f_mlo + torch.tanh(self.pyramid_conv(f_mlo))
            
        if self.has_attention or self.is_hybrid_fusion:
            f_cc_aligned = self.attn_mlo_to_cc(f_cc, f_mlo)
            f_mlo_aligned = self.attn_cc_to_mlo(f_mlo, f_cc)
            f_cc, f_mlo = f_cc_aligned, f_mlo_aligned
            
        if self.is_hybrid_fusion or self.mode == 'Late_Fusion':
            logit_cc = self.head_cc(f_cc).squeeze(-1)
            logit_mlo = self.head_mlo(f_mlo).squeeze(-1)
            logit_final = (logit_cc + logit_mlo) / 2
            
            out = {'logit_bin': logit_final}
            if self.training:
                out['aux_cc'] = logit_cc
                out['aux_mlo'] = logit_mlo
        else:
            f_fused = torch.cat([f_cc, f_mlo], dim=-1)
            logits = self.classifier(f_fused)
            
            out = {'logit_bin': logits.squeeze(-1)}
            if self.has_aux and self.training:
                out['aux_cc'] = self.aux_cc(f_cc).squeeze(-1)
                out['aux_mlo'] = self.aux_mlo(f_mlo).squeeze(-1)
            
        return out

# ─── 4. CORE TRAINING ENGINE WITH DISCRIMINATIVE POLICY ───────────────
def train_fold(df_task, label_col, fold_k, mode, task_name):
    print(f"\n  --> Training Started for Outer Fold {fold_k} | Variant Mode: {mode}")
    print("  " + "-"*85)
    train_tf, val_tf = get_transforms()
    
    # Isolated outer held-out validation mapping 
    df_outer_test = df_task[df_task['fold'] == fold_k].reset_index(drop=True)
    df_outer_train = df_task[df_task['fold'] != fold_k].copy()
    
    # Deterministic dynamic inner validation fold generation
    available_folds = [f for f in range(5) if f != fold_k]
    inner_val_fold = available_folds[-1]  
    
    df_inner_val   = df_outer_train[df_outer_train['fold'] == inner_val_fold].reset_index(drop=True)
    df_inner_train = df_outer_train[df_outer_train['fold'] != inner_val_fold].reset_index(drop=True)
    
    # Setup data loaders
    counts = np.bincount(df_inner_train[label_col].values, minlength=2)
    weights = (1.0 / np.maximum(counts, 1))[df_inner_train[label_col].values]
    sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), len(df_inner_train), replacement=True)
    
    tr_ld = DataLoader(DualViewDataset(df_inner_train, train_tf, label_col), BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va_ld = DataLoader(DualViewDataset(df_inner_val, val_tf, label_col), BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    te_ld = DataLoader(DualViewDataset(df_outer_test, val_tf, label_col), BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = MammoXVAblationModel(mode=mode).to(DEVICE)
    
    # Discriminative Learning Rate Policy implementation — Blocks feature destruction
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if 'backbone' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)
            
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5, 'weight_decay': 1e-2}, # Protects pretrained weights
        {'params': head_params, 'lr': 1e-4, 'weight_decay': 1e-2}     # Drives projection mapping
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-7)
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler()
    
    best_inner_auc, patience_ctr = -1.0, 0
    ckpt_path = os.path.join(CKPT_DIR, f"ablation_{task_name}_{mode}_fold{fold_k}.pt")
    
    # Structural training execution loop
    for epoch in range(1, NUM_EPOCHS + 1):
        frozen = epoch <= WARMUP_EPOCHS
        for p in model.backbone.parameters():
            p.requires_grad = not frozen
            
        model.train()
        running_loss = 0.0
        for batch in tr_ld:
            optimizer.zero_grad()
            cc_inputs = batch['cc'].to(DEVICE)
            mlo_inputs = batch['mlo'].to(DEVICE)
            labels = batch['label'].to(DEVICE).float()
            
            with autocast():
                outputs = model(cc_inputs, mlo_inputs)
                loss = criterion(outputs['logit_bin'], labels)
                
                if 'aux_cc' in outputs:
                    loss_cc = criterion(outputs['aux_cc'], labels)
                    loss_mlo = criterion(outputs['aux_mlo'], labels)
                    loss = loss + 0.2 * (loss_cc + loss_mlo)
                    
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        if not frozen:
            scheduler.step()
            
        # Inner Evaluation pass tracking loop
        model.eval()
        va_p, va_y = [], []
        with torch.no_grad():
            for batch in va_ld:
                outputs = model(batch['cc'].to(DEVICE), batch['mlo'].to(DEVICE))
                probs = torch.sigmoid(outputs['logit_bin']).cpu().numpy()
                va_p.extend(probs)
                va_y.extend(batch['label'].numpy())
                
        inner_va_auc = roc_auc_score(va_y, va_p) if len(np.unique(va_y)) > 1 else 0.0
        
        improved = inner_va_auc > best_inner_auc
        if improved:
            best_inner_auc = inner_va_auc
            patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
        elif epoch >= ES_START:
            patience_ctr += 1
            
        # Explicit 'Inner Val AUC' print verification statement for peer review tracking
        status = "Warmup" if frozen else "Active"
        marker = "*" if improved else " "
        print(f"    Epoch {epoch:02d}/{NUM_EPOCHS} [{status}] | Train Loss: {running_loss/len(tr_ld):.4f} | "
              f"Inner Val AUC: {inner_va_auc:.4f} | Best Inner AUC: {best_inner_auc:.4f} {marker} | Patience: {patience_ctr}/{PATIENCE}")
            
        if epoch >= ES_START and patience_ctr >= PATIENCE:
            print(f"    [EARLY STOP] Triggered at epoch {epoch}. Model converged based on Inner Validation.")
            break
            
    # Uncompromised outer verification extraction scoring execution
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    te_p, te_y = [], []
    with torch.no_grad():
        for batch in te_ld:
            outputs = model(batch['cc'].to(DEVICE), batch['mlo'].to(DEVICE))
            probs = torch.sigmoid(outputs['logit_bin']).cpu().numpy()
            te_p.extend(probs)
            te_y.extend(batch['label'].numpy())
            
    outer_test_auc = roc_auc_score(te_y, te_p) if len(np.unique(te_y)) > 1 else 0.0
    
    # Explicit print of uncompromised outer test metrics per fold closure
    print(f"  --> Outer Fold {fold_k} Complete.")
    print(f"      [Final Metrics] Best Inner Val AUC: {best_inner_auc:.4f} | Pristine Outer Test AUC: {outer_test_auc:.4f}\n")
    return df_outer_test['patient_id'].values, np.array(te_y), np.array(te_p)


# ─── 5. METRIC COMPUTATION PLUGINS (MANUSCRIPT STANDARDS) ────────────
def fixed_point_clinical_metrics(y_true, y_pred, target_spec=0.90, target_sens=0.95):
    """Computes specialized clinical operating points mandated by radiology journals."""
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    
    # Sens@Spec90: Find sensitivity where specificity >= target_spec
    sens_at_spec = 0.0
    for f, t in zip(fpr, tpr):
        if (1 - f) >= target_spec:
            sens_at_spec = max(sens_at_spec, t)
            
    # Spec@Sens95: Find specificity where sensitivity >= target_sens
    spec_at_sens = 0.0
    for f, t in zip(fpr, tpr):
        if t >= target_sens:
            spec_at_sens = max(spec_at_sens, (1 - f))
            
    return sens_at_spec, spec_at_sens


# ─── 6. MAIN PIPELINE EXECUTION SUITE ─────────────────────────────────
if __name__ == "__main__":
    csv_input_path = os.path.join(SPLIT_DIR, "patient_folds.csv")
    if not os.path.exists(csv_input_path):
        raise FileNotFoundError(f"Upstream dependencies missing. Run 02_data_pipeline.ipynb first: {csv_input_path}")
        
    df_main = pd.read_csv(csv_input_path)
    df_t1 = df_main[df_main['label_char'] != -1].reset_index(drop=True)
    
    # Unbiased structural modeling evaluation registries
    ABLATION_MODES = [
        'Pyramid_CrossAttn_JointHead',
        'Pyramid_CrossAttn_LateFusion',
        'Late_Fusion',
        'Ablation_No_Attention_GAP',
        'Ablation_Single_Scale',
        'Ablation_No_Aux_Loss'
    ]
    
    ablation_summary_rows = []
    
    for mode in ABLATION_MODES:
        print(f"\n{'='*85}\n  BENCHMARK RUNNING: {mode}\n{'='*85}")
        all_patient_ids, all_true_labels, all_pred_probs = [], [], []
        
        for fold_idx in range(5):
            p_ids, true_y, pred_p = train_fold(df_t1, 'label_char', fold_idx, mode, 't1_char')
            all_patient_ids.extend(p_ids)
            all_true_labels.extend(true_y)
            all_pred_probs.extend(pred_p)
            
        y_true_arr = np.array(all_true_labels)
        y_pred_arr = np.array(all_pred_probs)
        
        # Calculate uncompromised global statistical metrics
        oof_auc = roc_auc_score(y_true_arr, y_pred_arr)
        sens_at_sp90, spec_at_se95 = fixed_point_clinical_metrics(y_true_arr, y_pred_arr)
        
        print(f"\n📈 Consolidated Matrix Report for {mode}:")
        print(f"   Final Leakage-Free Global OOF AUC = {oof_auc:.4f}")
        
        # Save OOF Predictions
        df_oof = pd.DataFrame({'patient_id': all_patient_ids, 'label': all_true_labels, 'prob': all_pred_probs})
        df_oof.to_csv(os.path.join(STAT_DIR, f"oof_ablation_t1_char_{mode}.csv"), index=False)
        
        ablation_summary_rows.append({
            'Architecture Mode / Variant': mode,
            'OOF-AUC': f"{oof_auc:.4f}",
            'Sens @ Spec >= 0.90': f"{sens_at_sp90:.4f}",
            'Spec @ Sens >= 0.95': f"{spec_at_se95:.4f}"
        })
        
    # GENERATE PERFECT MARKDOWN TABLE READY FOR MANUSCRIPT INSERTION
    df_results = pd.DataFrame(ablation_summary_rows).sort_values(by='OOF-AUC', ascending=False).reset_index(drop=True)
    
    print("\n" + "="*105)
    print(" TABLE: STRUCTURAL ABLATION MATRIX (TASK 1 - CHARACTERIZATION, N=664) ")
    print("="*105)
    print(df_results.to_string(index=False))
    print("="*105)
    
    df_results.to_csv(os.path.join(STAT_DIR, 'ablation_study_results.csv'), index=False)
    print(f"\n Manuscript table successfully compiled and serialized to → {STAT_DIR}/ablation_study_results.csv")

Device: cuda | GPU: NVIDIA GeForce RTX 5090

  BENCHMARK RUNNING: Pyramid_CrossAttn_JointHead

  --> Training Started for Outer Fold 0 | Variant Mode: Pyramid_CrossAttn_JointHead
  -------------------------------------------------------------------------------------
    Epoch 01/50 [Warmup] | Train Loss: 1.1334 | Inner Val AUC: 0.6527 | Best Inner AUC: 0.6527 * | Patience: 0/12
    Epoch 02/50 [Warmup] | Train Loss: 0.9292 | Inner Val AUC: 0.7014 | Best Inner AUC: 0.7014 * | Patience: 0/12
    Epoch 03/50 [Warmup] | Train Loss: 0.9176 | Inner Val AUC: 0.7132 | Best Inner AUC: 0.7132 * | Patience: 0/12
    Epoch 04/50 [Active] | Train Loss: 0.8336 | Inner Val AUC: 0.7642 | Best Inner AUC: 0.7642 * | Patience: 0/12
    Epoch 05/50 [Active] | Train Loss: 0.7765 | Inner Val AUC: 0.7630 | Best Inner AUC: 0.7642   | Patience: 0/12
    Epoch 06/50 [Active] | Train Loss: 0.6987 | Inner Val AUC: 0.7817 | Best Inner AUC: 0.7817 * | Patience: 0/12
    Epoch 07/50 [Active] | Train Loss: 0.6927 | I

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f231319e320>
Traceback (most recent call last):
  File "/home/user/miniconda3/envs/fast/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/user/miniconda3/envs/fast/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/home/user/miniconda3/envs/fast/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f231319e320>
Traceback (most recent call last):
  File "/home/user/miniconda3/envs/fast/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/user/miniconda3/envs/fast/lib/python3.10/site-p

    Epoch 08/50 [Active] | Train Loss: 0.6608 | Inner Val AUC: 0.8084 | Best Inner AUC: 0.8084 * | Patience: 0/12
    Epoch 09/50 [Active] | Train Loss: 0.6546 | Inner Val AUC: 0.8176 | Best Inner AUC: 0.8176 * | Patience: 0/12
    Epoch 10/50 [Active] | Train Loss: 0.6516 | Inner Val AUC: 0.8066 | Best Inner AUC: 0.8176   | Patience: 1/12
    Epoch 11/50 [Active] | Train Loss: 0.5423 | Inner Val AUC: 0.8000 | Best Inner AUC: 0.8176   | Patience: 2/12
    Epoch 12/50 [Active] | Train Loss: 0.5526 | Inner Val AUC: 0.7895 | Best Inner AUC: 0.8176   | Patience: 3/12
    Epoch 13/50 [Active] | Train Loss: 0.4960 | Inner Val AUC: 0.8064 | Best Inner AUC: 0.8176   | Patience: 4/12
    Epoch 14/50 [Active] | Train Loss: 0.4596 | Inner Val AUC: 0.8155 | Best Inner AUC: 0.8176   | Patience: 5/12
    Epoch 15/50 [Active] | Train Loss: 0.2883 | Inner Val AUC: 0.7881 | Best Inner AUC: 0.8176   | Patience: 6/12
    Epoch 16/50 [Active] | Train Loss: 0.4379 | Inner Val AUC: 0.8221 | Best Inner AUC: 

In [17]:
# =====================================================================
# Dual-View Mammography Deep Learning Framework
# Task 1 — Architectural Ablation Summary Evaluation Suite
#
# Core Focus: Single-cell inference engine for ablation matrix only.
# Parses saved OOF CSV files to synthesize a publication-grade table.
# Bypasses model retraining loops entirely. English logging enforced.
# =====================================================================

import os
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5

# Paths Configuration
SPLIT_CSV = './splits/patient_folds.csv'
STAT_DIR  = './outputs/statistics'

def compute_clinical_metrics(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    # Optimal threshold selection via Youden's J Index
    t = thr[np.argmax(tpr - fpr)] 
    preds = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        'auc': auc, 
        'sens': tp / (tp + fn + 1e-8), 
        'spec': tn / (tn + fp + 1e-8)
    }

def bootstrap_auc_ci(y_true, y_pred, n_bootstraps=1000):
    rng = np.random.default_rng(SEED)
    aucs = []
    for _ in range(n_bootstraps):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2: 
            continue
        aucs.append(roc_auc_score(y_true[idx], y_pred[idx]))
    return np.percentile(aucs, [2.5, 97.5])

if __name__ == "__main__":
    if not os.path.exists(SPLIT_CSV):
        raise FileNotFoundError(f"Missing patient split reference file: {SPLIT_CSV}")
        
    df_main = pd.read_csv(SPLIT_CSV)
    df_t1 = df_main[df_main['label_char'] != -1].reset_index(drop=True)
    
    ABLATION_MODES = [
        'Ablation_No_Attention_GAP', 
        'Late_Fusion', 
        'Ablation_Single_Scale',
        'Pyramid_CrossAttn_JointHead', 
        'Pyramid_CrossAttn_LateFusion', 
        'Ablation_No_Aux_Loss'
    ]
    
    abl_records = []
    print("Extracting cached ablation traces from disk logs...")
    
    for mode in ABLATION_MODES:
        abl_path = os.path.join(STAT_DIR, f"oof_ablation_t1_char_{mode}.csv")
        if not os.path.exists(abl_path):
            print(f"  [WARN] Missing prediction file for mode: {mode} at {abl_path}")
            continue
        
        df_a = pd.read_csv(abl_path)
        yt_a, yp_a = df_a['label'].values.astype(float), df_a['prob'].values.astype(float)
        
        m_a = compute_clinical_metrics(yt_a, yp_a)
        lo_a, hi_a = bootstrap_auc_ci(yt_a, yp_a)
        
        # Calculate Mean Fold metrics across validation splits
        f_scores_a = []
        for k in range(N_FOLDS):
            pids_f = df_t1[df_t1['fold'] == k]['patient_id'].values
            oof_f = df_a[df_a['patient_id'].isin(pids_f)]
            if len(oof_f) > 0 and len(np.unique(oof_f['label'].values)) > 1:
                f_scores_a.append(roc_auc_score(oof_f['label'].values, oof_f['prob'].values))
                
        abl_records.append({
            'Architecture Mode / Variant': mode,
            'OOF-AUC': m_a['auc'],
            'Mean Fold-AUC': f"{np.mean(f_scores_a):.4f} (±{np.std(f_scores_a):.4f})" if f_scores_a else "0.0000",
            '95% Confidence Interval': f"[{lo_a:.3f} - {hi_a:.3f}]",
            'Sensitivity': f"{m_a['sens']:.3f}",
            'Specificity': f"{m_a['spec']:.3f}"
        })
        
    print("\n" + "="*115)
    print(" TABLE: STRUCTURAL ABLATION SYSTEM MATRIX (TASK 1 - CHARACTERIZATION, N=664) ")
    print("="*115)
    
    if abl_records:
        df_abl_rendered = pd.DataFrame(abl_records).sort_values('OOF-AUC', ascending=False).reset_index(drop=True)
        df_abl_rendered['OOF-AUC'] = df_abl_rendered['OOF-AUC'].map(lambda x: f"{x:.4f}")
        
        # Print perfectly formatted matrix directly to terminal/notebook cell output
        print(df_abl_rendered.to_string(index=False))
        
        # Save standardized summary table back to storage tracking paths
        output_csv_path = os.path.join(STAT_DIR, 'ablation_study_results.csv')
        df_abl_rendered.to_csv(output_csv_path, index=False)
        print(f"\n  [SUCCESS] Ablation summary compiled and saved to → {output_csv_path}")
    else:
        print("  [ERROR] No cached ablation metrics or CSV traces detected on disk.")
    print("="*115 + "\n")

Extracting cached ablation traces from disk logs...

 TABLE: STRUCTURAL ABLATION SYSTEM MATRIX (TASK 1 - CHARACTERIZATION, N=664) 
 Architecture Mode / Variant OOF-AUC    Mean Fold-AUC 95% Confidence Interval Sensitivity Specificity
   Ablation_No_Attention_GAP  0.8094 0.8114 (±0.0339)         [0.775 - 0.841]       0.684       0.801
                 Late_Fusion  0.7978 0.8013 (±0.0196)         [0.761 - 0.830]       0.654       0.828
       Ablation_Single_Scale  0.7880 0.8010 (±0.0221)         [0.755 - 0.819]       0.632       0.808
 Pyramid_CrossAttn_JointHead  0.7874 0.7930 (±0.0496)         [0.749 - 0.822]       0.747       0.694
Pyramid_CrossAttn_LateFusion  0.7753 0.7849 (±0.0394)         [0.740 - 0.810]       0.668       0.801
        Ablation_No_Aux_Loss  0.7744 0.7865 (±0.0344)         [0.736 - 0.806]       0.605       0.855

  [SUCCESS] Ablation summary compiled and saved to → ./outputs/statistics/ablation_study_results.csv

